# pix2pix 튜토리얼

## 핵심 질문
- enc8이 단독인데 왜 forward에서는 enc8과 enc7이 함께 처리되나요?


### 1. 단독의 의미
- 단독 = 이전 decoder 없이 시작
- 단독 != skip 없이

### 2. UpConvBlock은 항상 2개 입력
- 모든 UpConvBlock의 forward는 x와 skip 2개를 받습니다.
- enc8도 예외가 아닙니다.

### 3. 처리 과정
- Step 1: enc8 입력
- Step 2: enc8 Upsample
- Step 3: enc7과 Concat
- Step 4: Conv 처리

### 4. 건물 비유
- 지하(enc8)에서 시작하지만, 7층(enc7) 물건을 챙깁니다.
이것이 Skip Connection입니다.
---


## 1. Import


In [1]:
import torch
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


Device: cuda


## 2. 핵심 이해
### 구조 선언 시
```python
self.up_conv_layer1 = UpConvBlock(512, 512)
```
### Forward 사용 시
```python
dec1 = self.up_conv_layer1(enc8, enc7)
```
2개 입력을 받습니다!
---


## 3. DownConvBlock


In [2]:
class DownConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, norm=True, dropout=0.0):
        super(DownConvBlock, self).__init__()

        layers = []
        layers.append(nn.Conv2d(in_channels, out_channels, 4, 2, 1,
                                bias=False if norm else True))
        # 4: 4*4 kernel, stride=2, padding=1
        if norm:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2, inplace=True))
        if dropout > 0:
            layers.append(nn.Dropout(dropout))

        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)
print("DownConvBlock 정의 완료")


DownConvBlock 정의 완료


## 4. UpConvBlock (핵심!)


In [3]:
class UpConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, skip_channels, dropout=0.0):
        super(UpConvBlock, self).__init__()

        self.up = nn.ConvTranspose2d(in_channels, out_channels, 4, 2, 1, bias=False)
        self.conv = nn.Conv2d(out_channels + skip_channels, out_channels, 3, 1, 1, bias=False)
        self.norm = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.use_dropout = dropout > 0
        if self.use_dropout:
            self.dropout = nn.Dropout(dropout)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        x = self.conv(x)
        x = self.norm(x)
        x = self.relu(x)
        if self.use_dropout:
            x = self.dropout(x)
        return x
print("UpConvBlock 정의 완료")


UpConvBlock 정의 완료


## 5. enc8-enc7 처리 확인


In [4]:
enc8 = torch.randn(1, 512, 1, 1)
enc7 = torch.randn(1, 512, 2, 2)
up1 = UpConvBlock(512, 512, skip_channels=512, dropout=0.5)
print("입력:")
print(f"  enc8: {enc8.shape}")
print(f"  enc7: {enc7.shape}")
with torch.no_grad():
    x_up = up1.up(enc8)
    print(f"\nUpsample:")
    print(f"  {enc8.shape} -> {x_up.shape}")

    x_cat = torch.cat([x_up, enc7], dim=1)
    print(f"\nConcat:")
    print(f"  {x_up.shape} + {enc7.shape} = {x_cat.shape}")

    dec1 = up1(enc8, enc7)
    print(f"\n최종:")
    print(f"  dec1: {dec1.shape}")


입력:
  enc8: torch.Size([1, 512, 1, 1])
  enc7: torch.Size([1, 512, 2, 2])

Upsample:
  torch.Size([1, 512, 1, 1]) -> torch.Size([1, 512, 2, 2])

Concat:
  torch.Size([1, 512, 2, 2]) + torch.Size([1, 512, 2, 2]) = torch.Size([1, 1024, 2, 2])

최종:
  dec1: torch.Size([1, 512, 2, 2])


## 6. U-Net Generator


In [5]:
class UNetGenerator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super(UNetGenerator, self).__init__()

        # Encoder
        self.down1 = DownConvBlock(in_channels, 64, norm=False)
        self.down2 = DownConvBlock(64, 128)
        self.down3 = DownConvBlock(128, 256)
        self.down4 = DownConvBlock(256, 512, dropout=0.5)
        self.down5 = DownConvBlock(512, 512, dropout=0.5)
        self.down6 = DownConvBlock(512, 512, dropout=0.5)
        self.down7 = DownConvBlock(512, 512, dropout=0.5)
        self.down8 = DownConvBlock(512, 512, norm=False, dropout=0.5)

        # Decoder
        self.up1 = UpConvBlock(512, 512, skip_channels=512, dropout=0.5)
        self.up2 = UpConvBlock(512, 512, skip_channels=512, dropout=0.5)
        self.up3 = UpConvBlock(512, 512, skip_channels=512, dropout=0.5)
        self.up4 = UpConvBlock(512, 512, skip_channels=512, dropout=0.5)
        self.up5 = UpConvBlock(512, 256, skip_channels=256)
        self.up6 = UpConvBlock(256, 128, skip_channels=128)
        self.up7 = UpConvBlock(128, 64, skip_channels=64)

        # Final
        self.final = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, out_channels, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        # Encoder
        enc1 = self.down1(x)
        enc2 = self.down2(enc1)
        enc3 = self.down3(enc2)
        enc4 = self.down4(enc3)
        enc5 = self.down5(enc4)
        enc6 = self.down6(enc5)
        enc7 = self.down7(enc6)
        enc8 = self.down8(enc7) # bottle neck

        # Decoder (모두 2개 입력!)
        # 중요: enc8과 enc7 함께 입력해서 시작한다.
        dec1 = self.up1(enc8, enc7)
        dec2 = self.up2(dec1, enc6)
        dec3 = self.up3(dec2, enc5)
        dec4 = self.up4(dec3, enc4)
        dec5 = self.up5(dec4, enc3)
        dec6 = self.up6(dec5, enc2)
        dec7 = self.up7(dec6, enc1)

        return self.final(dec7)
print("U-Net Generator 정의 완료")


U-Net Generator 정의 완료


## 7. 테스트


In [6]:
model = UNetGenerator().to(device)
test_input = torch.randn(2, 3, 256, 256).to(device)
print(f"입력: {test_input.shape}")
with torch.no_grad():
    output = model(test_input)
print(f"출력: {output.shape}")
print(f"범위: [{output.min():.3f}, {output.max():.3f}]")
params = sum(p.numel() for p in model.parameters())
print(f"\n파라미터: {params:,}")


입력: torch.Size([2, 3, 256, 256])
출력: torch.Size([2, 3, 256, 256])
범위: [-1.000, 1.000]

파라미터: 59,498,691


## 8. Q&A

### Q1. enc8 단독의 의미는?
- 이전 decoder 없이 시작한다는 의미입니다.
- skip 없이가 아닙니다.

### Q2. 왜 forward에서는 2개를 받나요?
- UpConvBlock의 forward는 항상 x와 skip 2개를 받습니다.
- enc8도 예외가 아닙니다.

### Q3. U-Net의 핵심은?
- Skip Connection입니다.
- 모든 decoder가 encoder의 정보를 받습니다.
---


## 9. 핵심 요약

### 반드시 기억할 것
1. 단독 = 이전 decoder 없이
2. enc8 = Bottleneck, 시작점
3. forward(x, skip) 2개 입력 필수
4. Skip Connection = U-Net 핵심

### 한 줄 요약
- enc8은 시작점이지만 enc7과 함께 처리됩니다!
